### Import the Libraries and set the style for Plotting

In [ ]:
import torch, torch.nn as nn
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [ ]:
plt.style.use('dark_background')

### Load the Dataset

##### Training Set
- Input parameters will be:
    - demand at every one of the 1242 junctions
    - pressure at every one of the 1242 junctions
    - flowrate through every one of the 1270 pipes
    - time_of_day

- Therefore the total number of input dimension must be:
$$
1242 + 1242 + 1270 + 1 = 3755
$$
- The output field is leak intensity, i.e. Emitter Coefficients for every leaking node (Last Column of dataset)
- we standardize the variables so that one of them doesn't end up dominating over others when computing the gradients
$$
Z_\text{train}=\frac{X_{\text{train}}-\mu_{\text{train}}}{\sigma_{\text{train}}}
$$
this makes sure that all variables have $\text{mean}=0$ and $\text{standard deviation}=1$

In [ ]:
train_dataset = pd.read_csv(r"C:\Data\pipeline-leakage-system\Phase 1\pipeline_leak_dataset_expanded(train).csv")
X_train = train_dataset.iloc[:, :3755].fillna(0)
train_std = X_train.std()
train_mean = X_train.mean()
X_train = (X_train - train_mean) / train_std
x_train = torch.tensor(X_train.values, dtype=torch.float32)
train_dataset.head()

In [ ]:
y_train = torch.tensor(train_dataset.iloc[:, 3756:])
y_train = y_train.float()

##### Test Set

The training set is what the model learns from. The test data must then be put into the same coordinate system before being given to the trained model. so we will standardize the test set using the mean and variance of the training set and not the test set.
$$
Z_\text{test}=\frac{X_{\text{test}}-\mu_{\text{train}}}{\sigma_{\text{train}}}
$$

In [ ]:
test_dataset = pd.read_csv(r"C:\Data\pipeline-leakage-system\Phase 1\pipeline_leak_dataset_expanded(test).csv")
X_test = test_dataset.iloc[:, :3755].fillna(0)
X_test = (X_test - train_mean) / train_std
x_test = torch.tensor(X_test.values, dtype=torch.float32)
test_dataset.head()

In [ ]:
y_test = torch.tensor(test_dataset.iloc[:, 3756:])
y_test = y_train.float()

### Define the Deep Neural Network

In [ ]:
def DNN(D_i, D_k, D_o):
    arch = [D_i] + D_k + [D_o]
    layers = []
    for i in range(len(arch) - 1):
        layers.append(nn.Linear(arch[i], arch[i+1]))
        if i < len(arch) - 2:
            layers.append(nn.ReLU())
    model = nn.Sequential(*layers)
    return model

def network_output(x, model):
    out = model(x)
    D = out.shape[1] // 2
    logits = out[:, :D]      # per‑junction leak logits
    mu     = out[:, D:]      # per‑junction emitter coefficient
    return logits, mu

def heteroscedastic_network_output(x, model):
    out = model(x)
    D = out.shape[1] // 3
    logits  = out[:, :D]
    mu      = out[:, D:2 * D]
    log_var = out[:, 2 * D:]          
    return logits, mu, log_var

### Define Parameter Initialization

In [ ]:
def initialize_parameters(model):
    for index, layer in enumerate(model):
        if index % 2 == 0:
            nn.init.kaiming_normal_(layer.weight)
            layer.bias.data.fill_(0.0)
        print("Initializing Layer")

### Define Loss Function

##### Without $L2$ Regularization

In [ ]:
def Leak_Regression_Loss(x_train, network, y_train, alpha = 1.0, beta = 1.0):
    logits, mu = network_output(x_train, network)

    y_cls = (y_train > 0).float()
    cls_loss = torch.mean(
        torch.nn.functional.binary_cross_entropy_with_logits(
            logits, y_cls, reduction = 'none'
        )
    )

    p_leak   = torch.sigmoid(logits)
    reg_loss = torch.mean(p_leak * (y_train - mu) ** 2)

    return alpha * cls_loss + beta * reg_loss

def Leak_Heteroscedastic_Regression_Loss(x_train, network, y_train, alpha = 1.0, beta = 1.0):
    logits, mu, log_var = network_output(x_train, network)

    y_cls = (y_train > 0).float()
    cls_loss = torch.mean(
        torch.nn.functional.binary_cross_entropy_with_logits(
            logits, y_cls, reduction = 'none'
        )
    )

    p_leak   = torch.sigmoid(logits)
    var      = torch.exp(log_var)                       
    reg_loss = torch.mean(
        p_leak * (0.5 * log_var + (y_train - mu) ** 2 / (2.0 * var))
    )

    return alpha * cls_loss + beta * reg_loss

##### With $L2$ Regularization

In [ ]:
def L2_Leak_Regression_Loss(x_train, network, y_train, alpha = 1.0, beta = 1.0, lam = 1e-4):
    logits, mu = network_output(x_train, network)

    y_cls = (y_train > 0).float()
    cls_loss = torch.mean(
        torch.nn.functional.binary_cross_entropy_with_logits(
            logits, y_cls, reduction = 'none'
        )
    )

    p_leak   = torch.sigmoid(logits)
    reg_loss = torch.mean(p_leak * (y_train - mu) ** 2)

    l2_loss = sum(
        torch.sum(param ** 2)
        for name, param in network.named_parameters()
        if 'bias' not in name
    )

    return alpha * cls_loss + beta * reg_loss + lam * l2_loss

def L2_Leak_Heteroscedastic_Regression_Loss(x_train, network, y_train, alpha = 1.0, beta = 1.0, lam = 1e-4):
    logits, mu, log_var = network_output(x_train, network)

    y_cls = (y_train > 0).float()
    cls_loss = torch.mean(
        torch.nn.functional.binary_cross_entropy_with_logits(
            logits, y_cls, reduction = 'none'
        )
    )

    p_leak   = torch.sigmoid(logits)
    var      = torch.exp(log_var)                       
    reg_loss = torch.mean(
        p_leak * (0.5 * log_var + (y_train - mu) ** 2 / (2.0 * var))
    )

    l2_loss = sum(
        torch.sum(param ** 2)
        for name, param in network.named_parameters()
        if 'bias' not in name
    )

    return alpha * cls_loss + beta * reg_loss + lam * l2_loss

### Define Optimizers

##### Vanilla Gradient Descent

In [ ]:
def Gradient_Descent(x_train: torch.Tensor, y_train: torch.Tensor, x_test: torch.Tensor, y_test: torch.Tensor, model: nn.Sequential, loss_function: function, epochs: int, learning_rate = 0.01):
    train_losses = np.empty(epochs)
    test_losses = np.empty(epochs)
    for epoch in range(epochs):
        parameters = torch.nn.utils.parameters_to_vector(model.parameters())
        train_loss = loss_function(x_train, model, y_train)
        train_losses[epoch] = train_loss.item()
        test_loss = loss_function(x_test, model, y_test)
        test_losses[epoch] = test_loss.item()
        grads = torch.nn.utils.parameters_to_vector(torch.autograd.grad(train_loss, tuple(model.parameters())))
        with torch.no_grad():
            new_params = parameters - learning_rate * grads
            torch.nn.utils.vector_to_parameters(new_params, model.parameters())
    return train_losses, test_losses

##### Stochastic Gradient Descent

In [ ]:
def SGD(x_train: torch.Tensor, y_train: torch.Tensor, x_test: torch.Tensor, y_test: torch.Tensor, model: nn.Sequential, loss_function: function, epochs: int, learning_rate = 0.01, batch_size: int = 64):
    N = x_train.shape[0]
    train_losses = np.empty(epochs)
    test_losses = np.empty(epochs)
    for epoch in range(epochs):
        train_loss = loss_function(x_train, model, y_train)
        train_losses[epoch] = train_loss.item()
        test_loss = loss_function(x_test, model, y_test)
        test_losses[epoch] = test_loss.item()

        permutation = torch.randperm(N)
        for i in range(0, N, batch_size):
            idx = permutation[i:i + batch_size]
            x_batch, y_batch = x_train[idx], y_train[idx]

            batch_loss = loss_function(x_batch, model, y_batch)
            grads = torch.nn.utils.parameters_to_vector(torch.autograd.grad(batch_loss, tuple(model.parameters())))
            with torch.no_grad():
                parameters = torch.nn.utils.parameters_to_vector(model.parameters())
                new_params = parameters - learning_rate * grads
                torch.nn.utils.vector_to_parameters(new_params, model.parameters())
    return train_losses, test_losses

##### SGD with momentum

In [ ]:
def SGD_Momentum(x_train: torch.Tensor, y_train: torch.Tensor, x_test: torch.Tensor, y_test: torch.Tensor, model: nn.Sequential, loss_function: function, epochs: int, learning_rate = 0.01, batch_size: int = 64, beta = 0.9):
    N = x_train.shape[0]
    train_losses = np.empty(epochs)
    test_losses = np.empty(epochs)
    momentum = torch.zeros_like(torch.nn.utils.parameters_to_vector(model.parameters()))
    for epoch in range(epochs):
        train_loss = loss_function(x_train, model, y_train)
        train_losses[epoch] = train_loss.item()
        test_loss = loss_function(x_test, model, y_test)
        test_losses[epoch] = test_loss.item()

        permutation = torch.randperm(N)
        for i in range(0, N, batch_size):
            idx = permutation[i:i + batch_size]
            x_batch, y_batch = x_train[idx], y_train[idx]

            batch_loss = loss_function(x_batch, model, y_batch)
            grads = torch.nn.utils.parameters_to_vector(torch.autograd.grad(batch_loss, tuple(model.parameters())))
            momentum = beta * momentum + (1 - beta) * grads
            with torch.no_grad():
                parameters = torch.nn.utils.parameters_to_vector(model.parameters())
                new_params = parameters - learning_rate * momentum
                torch.nn.utils.vector_to_parameters(new_params, model.parameters())
    return train_losses, test_losses

##### RMSprop

In [ ]:
def RMSprop(x_train: torch.Tensor, y_train: torch.Tensor, x_test: torch.Tensor, y_test: torch.Tensor, model: nn.Sequential, loss_function: function, epochs: int, learning_rate = 0.001, batch_size: int = 64, rho = 0.9, eps = 1e-8):
    N = x_train.shape[0]
    train_losses = np.empty(epochs)
    test_losses = np.empty(epochs)
    v = torch.zeros_like(torch.nn.utils.parameters_to_vector(model.parameters()))
    for epoch in range(epochs):
        train_loss = loss_function(x_train, model, y_train)
        train_losses[epoch] = train_loss.item()
        test_loss = loss_function(x_test, model, y_test)
        test_losses[epoch] = test_loss.item()

        permutation = torch.randperm(N)
        for i in range(0, N, batch_size):
            idx = permutation[i:i + batch_size]
            x_batch, y_batch = x_train[idx], y_train[idx]

            batch_loss = loss_function(x_batch, model, y_batch)
            grads = torch.nn.utils.parameters_to_vector(torch.autograd.grad(batch_loss, tuple(model.parameters())))
            v = rho * v + (1 - rho) * grads ** 2
            with torch.no_grad():
                parameters = torch.nn.utils.parameters_to_vector(model.parameters())
                new_params = parameters - learning_rate * grads / (torch.sqrt(v) + eps)
                torch.nn.utils.vector_to_parameters(new_params, model.parameters())
    return train_losses, test_losses

##### Adam

In [ ]:
def Adam(x_train: torch.Tensor, y_train: torch.Tensor, x_test: torch.Tensor, y_test: torch.Tensor, model: nn.Sequential, loss_function: function, epochs: int, learning_rate = 0.001, batch_size: int = 64, beta = 0.9, gamma = 0.999, eps = 1e-8):
    N = x_train.shape[0]
    train_losses = np.empty(epochs)
    test_losses = np.empty(epochs)
    m = torch.zeros_like(torch.nn.utils.parameters_to_vector(model.parameters()))
    v = torch.zeros_like(m)
    t = 0
    for epoch in range(epochs):
        train_loss = loss_function(x_train, model, y_train)
        train_losses[epoch] = train_loss.item()
        test_loss = loss_function(x_test, model, y_test)
        test_losses[epoch] = test_loss.item()

        permutation = torch.randperm(N)
        for i in range(0, N, batch_size):
            idx = permutation[i:i + batch_size]
            x_batch, y_batch = x_train[idx], y_train[idx]

            batch_loss = loss_function(x_batch, model, y_batch)
            grads = torch.nn.utils.parameters_to_vector(torch.autograd.grad(batch_loss, tuple(model.parameters())))

            t += 1
            m = beta * m + (1 - beta) * grads
            v = gamma * v + (1 - gamma) * grads ** 2
            m_hat = m / (1 - beta ** t)
            v_hat = v / (1 - gamma ** t)

            with torch.no_grad():
                parameters = torch.nn.utils.parameters_to_vector(model.parameters())
                new_params = parameters - learning_rate * m_hat / (torch.sqrt(v_hat) + eps)
                torch.nn.utils.vector_to_parameters(new_params, model.parameters())
    return train_losses, test_losses